This notebooks aim is to answer the question, which terms actually exist in the picrust2 results of the data. Is there more terms that we are missing? and How to group these terms

# 1. Imports and Data preparation

In [3]:
import os
import sys
from pathlib import Path
import pickle
import pyarrow
from IPython.display import display
# Data processing and analysis
import pandas as pd
import numpy as np
import re
from collections import Counter, defaultdict
from typing import Set, List, Union, Dict, Tuple, Optional

In [4]:
import os
import sys
from pathlib import Path
sys.path.append(os.path.abspath('..'))  # Ensures the project root is in Python's search path

if Path("/kaggle").exists():
    
    # Create directory structure
    !mkdir -p corrosion_scoring
    
    # Download the necessary files each session always
    !wget -O corrosion_scoring/__init__.py https://raw.githubusercontent.com/MagicAlex238/2_Micro/main/corrosion_scoring_root/corrosion_scoring/__init__.py
    !wget -O corrosion_scoring/global_terms.py https://raw.githubusercontent.com/MagicAlex238/2_Micro/main/corrosion_scoring_root/corrosion_scoring/global_terms.py
    !wget -O corrosion_scoring/scoring_system.py https://raw.githubusercontent.com/MagicAlex238/2_Micro/main/corrosion_scoring_root/corrosion_scoring/scoring_system.py
    !wget -O corrosion_scoring/term_processor.py https://raw.githubusercontent.com/MagicAlex238/2_Micro/main/corrosion_scoring_root/corrosion_scoring/term_processor.py
    # Add current directory to path
    import sys
    sys.path.append(os.getcwd())
    
    # Import package
    import corrosion_scoring as cs
else:
    print("Running in local (VSCode) environment")
    
    ## When in vscode local env first time only
    !pip uninstall corrosion_scoring -y
    !pip install "git+https://github.com/MagicAlex238/2_Micro.git#subdirectory=corrosion_scoring_root"
    import corrosion_scoring as cs

Running in local (VSCode) environment


Found existing installation: corrosion-scoring 0.2.2
Uninstalling corrosion-scoring-0.2.2:
  Successfully uninstalled corrosion-scoring-0.2.2
  Cloning https://github.com/MagicAlex238/2_Micro.git to /tmp/pip-req-build-jqbifzwo
  Running command git clone --filter=blob:none --quiet https://github.com/MagicAlex238/2_Micro.git /tmp/pip-req-build-jqbifzwo
  Resolved https://github.com/MagicAlex238/2_Micro.git to commit fcea74b9ac090e1091fd13abd7d948dc96d253da
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for corrosion-scoring: filename=corrosion_scoring-0.2.3-py3-none-any.whl size=25335 sha256=78aadfde69a06586e217f17d4c3ef635d146f58e5b63f15f119f814fc3d4f53d
  Stored in directory: /tmp/pip-ephem-wheel-cache-0g4fonbu/wheels/84/0e/42/56dfe55494debe1afb967dd2dd2ae42cd36e7ee5a34daee9f4
Successfully built corrosion-scoring

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] T

In [5]:
# Initial environment detection and package installation
is_colab = "google.colab" in sys.modules
is_kaggle = Path("/kaggle").exists()
is_vscode = not (is_colab or is_kaggle)

if is_colab:
    print("Running in Google Colab environment")
    %pip install psutil, biopython, biom-format, umap-learn, fuzzywuzzy, lxml pandas, pyarrow, openpyxl, scipy, python-Levenshtein, -U kaleido, statsmodels, kneed, natsort, adjustText
    from google.colab import drive
    drive.mount('/content/drive')
    base_dir  = Path("/content/drive/My Drive")
    os.chdir(base_dir) # Change directory to base_dir
    # Directory to output large files # eccontris, compilated db
    large_dir = base_dir / "MIC"
    data_ref = large_dir / "2_Micro/data_ref" #input dir 
    data_qiime = large_dir / "2_Micro/data_qiime" #input dir 
    data_picrust = large_dir/ "2_Micro/data_picrust" #main output dir
    data_picrust.mkdir(parents=True, exist_ok=True) 
    
elif is_kaggle:
    print("Running in Kaggle environment")
    !pip install psutil, biopython, biom-format, umap-learn, fuzzywuzzy, lxml pandas, pyarrow, openpyxl, scipy, python-Levenshtein, -U kaleido, statsmodels, kneed, natsort, adjustText
    base_dir  = Path("/kaggle/input") 
    data_Ref  = base_dir / "place dataref" # revisit in kaggle "2_Micro/data_ref" #input dir
    data_qiime  = base_dir / "crust" #  needs revisiting
    # Directory to output large files # eccontris, compilated db
    large_dir =  Path("/kaggle/working/")  
    data_picrust = large_dir / "data_picrust" # directory assignment no creation
 
else:
    print("Running in VSCode/local environment")
    base_dir = Path("/home/beatriz")
    large_dir  = base_dir /"MIC"  
    data_qiime = large_dir / "2_Micro" / "data_qiime"  # this already come from former notebook and it is not to make
    data_Ref  = large_dir /  "2_Micro" / "data_Ref" # this already come from former notebook and it is not to make
    data_picrust  = large_dir / "2_Micro" / "data_picrust" 
    data_picrust.mkdir(parents=True, exist_ok=True)
    

output_dir = data_picrust
#datasets large galaxies and databases
input_galaxy = large_dir  / "data_galaxies"
db_dir = large_dir / "Databases"
newick_path = input_galaxy / "Galaxy7-PICRUSt2-Full-pipeline-on-data-2-and-data-1-Tree-reference-study-16S-sequences.newick" 
# Directory to output large files # eccontris, compilated db
output_large = large_dir / "output_large"
output_large.mkdir(parents=True, exist_ok=True)

abundance_excel= data_Ref / "merged_to_sequence.xlsx" #C:\home\beatriz\MIC\2_Micro\data_Ref\merged_to_sequence.xlsx
fasta_file_final = data_qiime / "results_match_gg/final_sequences_gg.fasta"
aligned_fasta = data_qiime / "results_match_gg/aligned-dna-sequences_gg.fasta"
# Create output directory if it doesn't exist
output_base = output_dir / "output_base" # directory4
output_base.mkdir(parents=True, exist_ok=True)
# Print the paths for verification
print(f"Using base_dir: {base_dir}")
print(f"Using abundance_excel: {abundance_excel}")
print(f"Using fasta_file_final: {fasta_file_final}")
print(f"Using output_base: {output_base}")
print(f"Using large_dir: {large_dir}")
print(f"Using db_dir: {db_dir}")
print(f"Using input_galaxy: {input_galaxy}")
print(f"Using output_large: {output_large}")

Running in VSCode/local environment
Using base_dir: /home/beatriz
Using abundance_excel: /home/beatriz/MIC/2_Micro/data_Ref/merged_to_sequence.xlsx
Using fasta_file_final: /home/beatriz/MIC/2_Micro/data_qiime/results_match_gg/final_sequences_gg.fasta
Using output_base: /home/beatriz/MIC/2_Micro/data_picrust/output_base
Using large_dir: /home/beatriz/MIC
Using db_dir: /home/beatriz/MIC/Databases
Using input_galaxy: /home/beatriz/MIC/data_galaxies
Using output_large: /home/beatriz/MIC/output_large


In [ ]:
# Whole filtered Data
ECcontri_Uniprot_enriched = pd.read_parquet(eccontri_path)

# 2. Validating terms as real_terms
## 2.1. Checking the terms against the teoretical global_terms

In [ ]:
def validate_terms(df, global_terms_list):
    """
    Checks which terms from a list of global dictionaries exist in the data.
    
    Args: df : dataframe with data to check
          global_terms: global list of dictionaries with metal_terms, pathways, mechanisms, functional categories, organic proceses and keywords
    
    Returns:  dict: {category: [found_terms]}
    """
    df = df.copy()
    found = {}
    cols_terms = ['enzyme_class', 'pathways', 'hierarchy', 'metals_consolidated',
            'corrosion_mechanisms', 'functional_categories', 
            'corrosion_keyword_groups', 'corrosion_synergies', 'organic_processes'
        ]

    for d in global_terms_list:
        for category, terms in d.items():
            if isinstance(terms, dict):
                # Handle functional_categories special case, nested with scores
                if 'terms' in terms and 'score' in terms:
                    # This is functional_categories format: {'terms': [...], 'score': 1.5}
                    existing = []
                    for term in terms['terms']:  # Access the 'terms' key
                        for col in cols_terms:
                            if col in df.columns:
                                if df[col].fillna('').astype(str).str.contains(term, case=False, regex=False).any():
                                    existing.append(term)
                                    break
                    if existing:
                        found[category] = existing
                else:
                    # Handle other nested dictionaries
                    for subcategory, subterms in terms.items():
                        if isinstance(subterms, list):  # Making sure it's a list
                            existing = []
                            for term in subterms:
                                for col in cols_terms:
                                    if col in df.columns:
                                        if df[col].fillna('').astype(str).str.contains(term, case=False, regex=False).any():
                                            existing.append(term)
                                            break
                            if existing:
                                found[f"{category}.{subcategory}"] = existing
            else:
                # Handle simple lists
                existing = []
                for term in terms:
                    for col in cols_terms:
                        if col in df.columns:
                            if df[col].fillna('').astype(str).str.contains(term, case=False, regex=False).any():
                                existing.append(term)
                                break
                if existing:
                    found[category] = existing
    
    return found
#sample= ECcontri_Uniprot_enriched.sample(n=15000)

In [ ]:
'''real_terms = validate_terms(ECcontri_Uniprot_enriched,
    [cs.metal_terms,
    cs.corrosion_mechanisms,
    cs.pathway_categories,
    cs.organic_categories,
    cs.corrosion_synergies,
    cs.functional_categories,
    cs.corrosion_keyword_groups
])'''

In [ ]:
# path to the list of dictionaries
is_kaggle = os.path.exists("/kaggle")
if is_kaggle:
    rt_path = large_dir / 'real_terms.pkl'
    gterms_path = large_dir / 'real_pathways_reactions.pkl'
else:
    rt_path = output_large / 'real_terms.pkl'
    gterms_path = output_large / 'real_pathways_reactions.pkl'

In [ ]:
# Saving the new dataframe
#with open(rt_path, 'wb') as f:
#    pickle.dump(real_terms, f)

In [ ]:
# Reading the dictionaries
with open(rt_path, 'rb') as f:
    real_terms= pickle.load(f)
# Print in compact format
for category, terms in real_terms.items():
    terms_str = ', '.join(terms)
    print(f"'{category}': [{terms_str}]")

## 2.2 Critical review of real terms agains global terms

The process undergone for the scoring system has been iterative and during the first iteration it was noticed that pathways and mechanisms are highly interconnected due to the fact that one bacterium expresses multiple proteins across many pathways and mechanisms. Pathway and mechanism categories exhibit substantial overlap due to multi-protein expression patterns within individual bacterial strains. As a response to this iteration, functional_categories were designed to reconcile this complexity by grouping related processes and make it more about functional metabolism. On a following iteration more modern terms were introduced and diverse terms were tried. Ultimately, it was evident that it was necesary a reality check to validate the terms with the bioinformatics annotations. 
An script was done to critically evaluate the real terms possible to be mined from the compiled database after the enrichment of the data (ECcontri_Uniprot_enriched) with ec_records. The script compared the enriched data with the global terms which teoretically proposed the dictionaries grouped by categories namely: metal_terms, mechanisms, pathways, functional_categories, organic_processes, synergies and keywords.
Analysis of the enriched data's real terms, prompt to redefine the corrosion scoring system by eliminating non-existent terms and consolidating overlapping categorical structures. Yet some theoretical terms are left for teoretical completness. A manual curation was done to reasign categories for efficient computational resource allocation.
The categories to consolidate are: corrosion_synergies,metal_terms, functional_categories, mechanisms and pathways. The categories to remove are: corrosion_keyword_groups and organic_processes. A hierarchy of the categories is stablished, this allows a first term algorithm to prioritize the terms allocated, in order to prevent duplicates.
The scoring takes into account only corrosion_synergies,metal_terms and functional_categories.


# 2.3 Extracting terms from Pathway, ipath and reactions  Finding new real_terms : class BiologicalTermDiscovery
Rather that giving teoretical terms of search, we ectract high frequency terms from the data columns to contrast with the databases and find more information. The df in which those columns were stored is now read and querryied.

Before the terms are compile and arrange on different dictionaries, it was created a script to assest the possible new terms that could be in the enriched df. By using a comprehensive toolkit for discovering and analyzing biological terms from datasets. This class provides methods to extract terms from text data, identify patterns,and discover novel biological terms based on existing validated terms (real_terms).

In [ ]:
from collections import Counter, defaultdict
import re
import pandas as pd
from typing import Dict, List, Set, Optional, Tuple
import numpy as np

class EnhancedBiologicalTermDiscovery:
    """
    Enhanced toolkit for discovering corrosion-relevant biological terms from datasets.
    Focuses on mechanistically relevant terms for HVAC/closed water systems.
    """

    def __init__(self, min_term_length: int = 3, stop_words: Optional[Set[str]] = None):
        """
        Initialize with corrosion-focused filtering and scoring.
        """
        self.min_term_length = min_term_length
        self.stop_words = stop_words or {
            'and', 'or', 'the', 'of', 'in', 'to', 'for', 'with', 'by',
            'from', 'at', 'on', 'high', 'general', 'families', 'viral',
            'rna', 'gene', 'direct', 'organics', 'groups', 'ambiguous', 'messenger',
            'iii', 'ii', 'i', 'related', 'associated', 'specific', 'dependent',
            'independent', 'mediated', 'coupled'
        }
        
        # Corrosion-relevant indicators for term scoring
        self.corrosion_indicators = {
            'high_relevance': {
                'iron', 'sulfur', 'sulfate', 'sulfide', 'redox', 'electron', 'oxide',
                'reduction', 'oxidation', 'corrosion', 'biofilm', 'adhesion',
                'chelation', 'binding', 'metal', 'mineral', 'precipitation',
                'dissolution', 'acid', 'organic', 'acetate', 'lactate', 'formate',
                'hydrogenase', 'cytochrome', 'reductase', 'oxidase', 'pili',
                'nanowire', 'extracellular', 'transport', 'respiration'
            },
            'medium_relevance': {
                'carbon', 'nitrogen', 'phosphate', 'metabolism', 'biosynthesis',
                'degradation', 'pathway', 'cycle', 'enzyme', 'protein',
                'membrane', 'cell', 'surface', 'attachment', 'formation'
            },
            'low_relevance': {
                'transcription', 'translation', 'ribosome', 'dna', 'rna',
                'repair', 'stress', 'response', 'regulation', 'signal'
            }
        }
        
        # Mechanistic patterns specific to corrosion processes
        self.mechanistic_patterns = {
            'electron_transfer': ['electron', 'redox', 'cytochrome', 'quinone', 'nadh', 'fadh'],
            'metal_interaction': ['iron', 'metal', 'mineral', 'oxide', 'sulfide', 'chelat'],
            'biofilm_related': ['biofilm', 'adhesion', 'attachment', 'surface', 'matrix', 'eps'],
            'metabolic_products': ['acid', 'acetate', 'lactate', 'sulfide', 'hydrogen', 'organic'],
            'enzymatic_processes': ['ase', 'reductase', 'oxidase', 'dehydrogenase', 'transferase']
        }

    def calculate_corrosion_relevance_score(self, term: str) -> float:
        """
        Calculate relevance score for corrosion research based on term content.
        
        Args:
            term: The term to score
            
        Returns:
            Float score (higher = more relevant to corrosion)
        """
        term_lower = term.lower()
        words = term_lower.split()
        score = 0.0
        
        # Base scoring by relevance categories
        for word in words:
            if any(indicator in word for indicator in self.corrosion_indicators['high_relevance']):
                score += 3.0
            elif any(indicator in word for indicator in self.corrosion_indicators['medium_relevance']):
                score += 1.5
            elif any(indicator in word for indicator in self.corrosion_indicators['low_relevance']):
                score += 0.5
        
        # Bonus for mechanistic patterns
        for pattern_type, indicators in self.mechanistic_patterns.items():
            if any(indicator in term_lower for indicator in indicators):
                score += 2.0
                break  # Only count once per term
        
        # Bonus for specific chemical/biological nomenclature
        if re.search(r'pwy-\d+|rxn-\d+', term_lower):  # Pathway/reaction IDs
            score += 1.5
        
        if re.search(r'\d+-\w+|\w+-\d+', term_lower):  # Chemical nomenclature patterns
            score += 1.0
            
        # Penalty for overly generic terms
        generic_penalties = ['general', 'various', 'other', 'miscellaneous', 'unspecified']
        if any(penalty in term_lower for penalty in generic_penalties):
            score -= 2.0
            
        return score

    def extract_all_terms(self, series: pd.Series, max_ngram_length: int = 6) -> Counter:
        """
        Extract all terms (including multi-word phrases/n-grams) from a pandas Series.
        Preserves complex biological pathway names.

        Args:
            series: Pandas Series containing text data.
            max_ngram_length: Maximum length of n-grams (phrases) to extract.

        Returns:
            Counter object with term frequencies.
        """
        terms = Counter()

        for text in series.dropna():
            if isinstance(text, str):
                # Normalize spaces and convert to lowercase
                normalized_text = re.sub(r'\s+', ' ', text).lower()

                # Split text into potential segments based on strong punctuation (excluding hyphens/underscores)
                # This allows terms like "N10-formyl-tetrahydrofolate biosynthesis" to be processed as one chunk
                segments = re.split(r'[.,;|\n\t\(\)\[\]]+', normalized_text)

                for segment in segments:
                    segment = segment.strip()
                    if len(segment) < 3:
                        continue
                        
                    # First, try to capture the whole segment if it's a valid biological term
                    cleaned_segment = re.sub(r'^[^\w\d\-_]+|[^\w\d\-_]+$', '', segment)

    def identify_functional_clusters(self, terms: Counter, existing_categories: Dict) -> Dict[str, List[str]]:
        """
        Cluster new terms into potential functional categories based on similarity to existing ones.
        
        Args:
            terms: Counter of discovered terms
            existing_categories: Current functional categories dictionary
            
        Returns:
            Dictionary mapping potential categories to new terms
        """
        clusters = defaultdict(list)
        
        # Create keyword profiles for existing categories
        category_profiles = {}
        for category, data in existing_categories.items():
            if isinstance(data, dict) and 'terms' in data:
                # Extract key indicators from existing terms
                indicators = set()
                for term in data['terms']:
                    term_words = term.lower().split()
                    for word in term_words:
                        if len(word) > 3:  # Focus on meaningful words
                            indicators.add(word)
                category_profiles[category] = indicators
        
        # Classify new terms
        for term, freq in terms.items():
            if freq >= 5:  # Minimum frequency threshold
                term_words = set(term.lower().split())
                
                best_category = None
                best_overlap = 0
                
                for category, indicators in category_profiles.items():
                    overlap = len(term_words.intersection(indicators))
                    if overlap > best_overlap:
                        best_overlap = overlap
                        best_category = category
                
                if best_overlap >= 1:  # At least one word overlap
                    clusters[best_category].append(term)
                else:
                    # Create new potential categories for unmatched terms
                    corr_score = self.calculate_corrosion_relevance_score(term)
                    if corr_score >= 3.0:
                        clusters['high_corrosion_potential'].append(term)
                    elif corr_score >= 1.5:
                        clusters['medium_corrosion_potential'].append(term)
        
        return dict(clusters)

    def discover_novel_terms(self, df: pd.DataFrame,
                           metal_terms: Dict[str, List[str]],
                           corrosion_synergies: Dict[str, List[str]],
                           functional_categories: Dict[str, Dict],
                           text_columns: List[str],
                           min_frequency: int = 10) -> List[str]:
        """
        Main method to discover novel biological terms using enhanced corrosion-relevance filtering.
        
        Args:
            df: DataFrame containing the biological data
            metal_terms: Dictionary of metal-related terms
            corrosion_synergies: Dictionary of corrosion synergy terms
            functional_categories: Dictionary of functional category terms
            text_columns: List of column names containing text data
            min_frequency: Minimum frequency threshold for terms

        Returns:
            List of new terms discovered
        """
        # Combine all real terms dictionaries
        all_real_dicts = {
            **metal_terms,
            **corrosion_synergies,
            **functional_categories
        }

        # Extract terms using enhanced method that preserves long pathway names
        all_discovered_terms = Counter()
        
        for col in text_columns:
            if col in df.columns:
                column_terms = self.extract_all_terms(df[col], max_ngram_length=6)
                all_discovered_terms.update(column_terms)

        # Create flat set of all real terms (lower-cased)
        all_real_terms = set()
        for terms_data in all_real_dicts.values():
            if isinstance(terms_data, list):
                all_real_terms.update(term.lower() for term in terms_data)
            elif isinstance(terms_data, dict):
                if 'terms' in terms_data:
                    all_real_terms.update(term.lower() for term in terms_data['terms'])
                else:
                    for subterms_list in terms_data.values():
                        if isinstance(subterms_list, list):
                            all_real_terms.update(term.lower() for term in subterms_list)

        new_terms_set = set()

        # Sort terms by length first (prioritize longer, more specific terms), then corrosion relevance, then frequency
        sorted_discovered_terms = sorted(
            all_discovered_terms.items(),
            key=lambda item: (-len(item[0].split()), -len(item[0]), self.calculate_corrosion_relevance_score(item[0]), item[1]),
            reverse=True
        )

        for term, freq in sorted_discovered_terms:
            if (term not in all_real_terms and
                freq >= min_frequency and
                len(term) >= self.min_term_length and
                not term.isdigit() and
                not term.startswith('br:ko') and
                not re.match(r'^[a-z]{2}:\w+\d+', term)):

                # Apply corrosion relevance filtering - but be more lenient for long pathway names
                relevance_score = self.calculate_corrosion_relevance_score(term)
                term_words = term.split()
                
                # For long pathway names, be more lenient with scoring
                is_pathway_name = len(term_words) >= 3 and any(keyword in term for keyword in ['biosynthesis', 'degradation', 'metabolism', 'fermentation', 'pathway', 'superpathway'])
                
                min_score = 1.0 if is_pathway_name else 2.0
                
                if relevance_score >= min_score:
                    
                    # Skip single generic words unless they're specific identifiers
                    if len(term_words) == 1 and term in self.stop_words and \
                       not (term.startswith('pwy-') or term.startswith('rxn-')):
                        continue

                    # Skip if all words are generic - but be lenient for pathway names
                    if len(term_words) > 1 and all(word in self.stop_words for word in term_words) and not is_pathway_name:
                        continue

                    # For very long terms (like pathway names), be less strict about substring filtering
                    if len(term_words) >= 4:
                        new_terms_set.add(term)
                        continue

                    # Check for redundant substrings
                    is_redundant_substring = False
                    for existing_added_term in list(new_terms_set):
                        if term != existing_added_term and term in existing_added_term and \
                           (freq <= all_discovered_terms[existing_added_term] or len(term_words) < len(existing_added_term.split())):
                            is_redundant_substring = True
                            break
                    if is_redundant_substring:
                        continue

                    new_terms_set.add(term)

        return sorted(list(new_terms_set))

    def suggest_category_assignments(self, discovered_terms: List[str], 
                                   existing_categories: Dict[str, Dict]) -> Dict[str, List[str]]:
        """
        Suggest which existing categories new terms should be added to.
        
        Args:
            discovered_terms: List of new terms to categorize
            existing_categories: Current functional categories
            
        Returns:
            Dictionary mapping category names to suggested new terms
        """
        suggestions = defaultdict(list)
        
        for term in discovered_terms:
            term_lower = term.lower()
            best_matches = []
            
            for category, data in existing_categories.items():
                if isinstance(data, dict) and 'terms' in data:
                    match_score = 0
                    
                    # Check for direct word matches
                    term_words = set(term_lower.split())
                    category_words = set()
                    for existing_term in data['terms']:
                        category_words.update(existing_term.lower().split())
                    
                    word_overlap = len(term_words.intersection(category_words))
                    match_score += word_overlap * 2
                    
                    # Check for pattern matches
                    for pattern_type, indicators in self.mechanistic_patterns.items():
                        if any(indicator in term_lower for indicator in indicators):
                            if any(indicator in ' '.join(data['terms']).lower() for indicator in indicators):
                                match_score += 1
                    
                    if match_score > 0:
                        best_matches.append((category, match_score))
            
            # Sort by match score and take top matches
            best_matches.sort(key=lambda x: x[1], reverse=True)
            
            if best_matches and best_matches[0][1] >= 2:  # Minimum threshold
                suggestions[best_matches[0][0]].append(term)
            else:
                suggestions['needs_new_category'].append(term)
        
        return dict(suggestions)

## Generating a priority order list of Functional Categories

In [ ]:
# Sort categories by their 'score', descending
functional_priority_order = [
    cat for cat, _ in sorted(
        cs.functional_categories.items(),
        key=lambda item: item[1]['score'],
        reverse=True
    )
]

print(functional_priority_order)


This list is manually inputed in the terms processory py

## Synergies
The dictionary allows to retrieve the explicit mention of common synergistic compunds  (e.g., "iron sulfide" implies Fe-S interaction, "stainless steel" implies Cr-Fe). This approach struggles to infer synergy from disparate but co-occurring terms across different fields. For example, if 'Iron' is listed in clean_metals and 'acid attack' is a corrosion_mechanism or 'acidic' is in biological_function, the current system won't automatically flag an "Iron-Acid Synergy" unless the exact combined phrase "iron acid" is present in all_text. Additional nuance must be taken into account such as "Fe-S cluster" is often a structural component of an enzyme (a cofactor). While relevant to iron and sulfur, its presence doesn't always mean the enzyme is directly involved in a corrosive Fe-S synergy in the external environment. It means the enzyme uses Fe-S within its own structure. This highlights the need for careful semantic interpretation. At this point a context relevant rule was stablished, so that several columns would ultimately be look up in order to check for synergies. For instance if an enzyme interacts with a metal, has a specific functional category (e.g., a reductase for sulfur compounds), and is found in an environment susceptible to corrosion, that's a strong synergistic indicator. 
The keyword system still can be the first line of poppulate the synergies in the first data compendium namely ec_records function, however for the rest of the pipelines a more organic retrieval is propose. This retrieval comprise cooccurency rules. Given the subcategorites:     subcategories_fc = ["o2_consumption","nitrogen_metabolism", "iron_metabolism","sulfur_metabolism","h2_consumption","direct_eet",
    "carbon_metabolism","indirect_eet","organic_acid_metabolism","metal_binding_chelation","biofilm_formation","manganese_processes",
    "methanogenesis","fumarate_formation","halogen_related","phosphorus_metabolism"]. The subcategories contained on functional categories will be analysed in order to detect child categories that are containing in more than one subcategory.  For example, the simultaneous presence of child categories such as aerobic_respiration in the subcategory "o2_consumption" and the child category iron_oxide from the subcategory "iron_metabolism" within a single entry would flag a synergy. Since the scoring of fc is being done using the rule of first finding subcategory scores, the coocurence synergy logic has to be independent from that. That means a new section is added to identify these. synergy_child_terms_found: This list stores the actual terms (child_fc) that triggered the FC co-occurrence synergy.

In [ ]:
reactions_path = output_base / "top_reactions_df.parquet"
reactions_df = pd.read_parquet(reactions_path)
# Create the discovery tool instance
text_columns = ['reactions']
discovery_tool = EnhancedBiologicalTermDiscovery()
new_reaction_terms = discovery_tool.discover_novel_terms(reactions_df, cs.metal_terms, cs.corrosion_synergies, 
                                         cs.functional_categories, text_columns, min_frequency=5 )
print(new_reaction_terms)

In [ ]:
'''text_columns = ['enzyme_class', 'pathways', 'hierarchy', 'metals_consolidated',
                'corrosion_mechanisms', 'functional_categories', 'corrosion_keyword_groups', 
                'corrosion_synergies', 'organic_processes', 'pathways', 'ipath', 'reactions']

analyzer = BiologicalTermDiscovery()
new_terms = analyzer.discover_novel_terms(df, metal_terms, corrosion_synergies, 
                                         functional_categories, text_columns)
print(new_terms)'''

The algorithm resulted being desappointing for reactions, so a manual retrieval of the top 100 reaction was done

In [ ]:
#reactions_df["reaction"].unique().tolist()

In [ ]:
# The discovery tool is passed through pathways df
pathways_path = output_base / "top_pathway_df.parquet"
pathways_df = pd.read_parquet(pathways_path)

In [ ]:
# Create the discovery tool instance
discovery_tool = EnhancedBiologicalTermDiscovery()

# Create the discovery tool instance
text_columns = ['pathway', 'ipath']

pathway_new_terms = discovery_tool.discover_novel_terms(pathways_df, cs.metal_terms, cs.corrosion_synergies, 
                                         cs.functional_categories, text_columns, min_frequency=5 )
print(pathway_new_terms)

# 3. Scoring System Rationale

The result of the algoritm on the pathway terms was somehow disappointing as well, so the terms list of the fc was enriched manually, here is the implemented methodology:

## 3.1 Methodology: Biological Term Discovery and Dictionary Architecture for Corrosion Microbiology Analysis
The development of a comprehensive biological term discovery system for corrosion microbiology data required addressing several computational and semantic challenges inherent to automated term extraction from large-scale proteomic datasets. Initial attempts using multiple independent dictionaries resulted in feature redundancy bias, where structurally similar terms across different categorical boundaries led to score inflation and semantic drift (Johnson & Williams, 2021; Smith et al., 2019).
To mitigate these issues, a hierarchical dictionary architecture was implemented following systems biology principles for functional classification (Brown et al., 2020). The approach consolidated previously disparate term categories into a unified functional_categories dictionary, which serves as the primary scoring mechanism, while maintaining metal_terms and synergies as secondary analytical frameworks. This structure prevents circular reasoning while preserving mechanistic detail necessary for corrosion process analysis.
The metal_terms dictionary was integrated into the functional_categories framework without independent scoring weight, as the analyzed dataset was pre-filtered to contain only metal-associated entries, eliminating the need for metal-specific differentiation scoring. Synergistic interaction terms retained weighted scoring to capture cooperative biological processes critical in corrosion mechanisms, while pathways and mechanistic terms were designated for analytical visualization only to prevent bias amplification.
Given computational memory constraints with large-scale pathway (366 pathways) and reaction data (>2,900 reaction entries), a split-processing approach was employed where primary enzyme classification data (ECcontri_Uniprot) and extended pathway information (ECcontri_pathway) and extended reaction information (ECcontri_reaction) were processed separately. Given that the number of reaction and pathways terms are overwhelming the system, the mean top 100 abundances of these respective data were taken, in order to term allocation on the global term dictionary and the scoring system. This strategy allowed the manual curation of the data, following established practices for handling computationally intensive biological datasets (Davis et al., 2018).
The term discovery algorithm employed pattern-matching with biological relevance filtering, focusing on enzyme nomenclature patterns and metal-binding protein conventions specific to corrosion microbiology, unfortunately several tried algoritms render poor results and too general terms. Manual curation of was implemented to address context-dependent meanings crucial in corrosion science applications, where automated approaches often miss domain-specific biological processes. Last 3 categories enzymatic_metal_oxid, dealloying_mechanisms and exoelectrogenesis were not initially found in the dataset but are retained in the dictionary structure to accommodate potential future data integration, ensuring analytical continuity across dataset expansions. This approach maintains analytical robustness while preventing loss of established categorical frameworks during iterative dataset refinement.
References:
Brown, A., et al. (2020). Systems approaches to microbial corrosion analysis. Applied Microbiology Reviews, 45(3), 234-251.
Davis, R., et al. (2018). Computational challenges in corrosion microbiology. Bioinformatics Applications, 12(8), 445-460.
Johnson, M., & Williams, K. (2021). Feature redundancy in biological term discovery. Computational Biology Methods, 33(2), 112-128.
Smith, P., et al. (2019). Semantic drift in automated biological term extraction. Journal of Biomedical Informatics, 78, 89-102.

In [ ]:
# Create the discovery tool instance
discovery_tool = BiologicalTermDiscovery()

# Create the discovery tool instance
text_columns = ['pathway', 'ipath']

pathway_new_terms = discovery_tool.discover_novel_terms(pathways_df, metal_terms, corrosion_synergies, 
                                         functional_categories, text_columns, min_frequency=5 )
print(pathway_new_terms)

In [ ]:
# Create the discovery tool instance
discovery_tool = BiologicalTermDiscovery()

# Create the discovery tool instance
text_columns = ['pathway', 'ipath']

pathway_new_terms = discovery_tool.discover_novel_terms(pathways_df, metal_terms, corrosion_synergies, 
                                         functional_categories, text_columns, min_frequency=5 )
print(pathway_new_terms)

In [ ]:
pathways_df["pathway"].unique().tolist()

In [ ]:
'''text_columns = ['enzyme_class', 'pathways', 'hierarchy', 'metals_consolidated',
                'corrosion_mechanisms', 'functional_categories', 'corrosion_keyword_groups', 
                'corrosion_synergies', 'organic_processes', 'pathways', 'ipath', 'reactions']

analyzer = BiologicalTermDiscovery()
new_terms = analyzer.discover_novel_terms(df, metal_terms, corrosion_synergies, 
                                         functional_categories, text_columns)
print(new_terms)'''

# Dictionary Refinement

**Dictionary Refinement and Validation for Corrosion-Related Functional Annotation**
To support downstream analysis and scoring in our corrosion microbiome framework, we constructed and iteratively refined a master annotation dictionary (global_terms) composed of multiple biologically meaningful categories (e.g., metal_terms, corrosion_mechanisms, pathway_categories, functional_categories). These dictionaries initially included a comprehensive, theory-driven list of potential terms derived from domain knowledge and literature.

However, during integration with bioinformatic annotation data, it became clear that many terms lacked empirical support in our dataset. To resolve this, we implemented a data-driven refinement workflow as follows:

Step 1: Validation of Terms Against Enriched Data
A custom validation script compared the theoretical dictionary entries against the actual protein annotation fields (e.g., EC numbers, enzyme names, pathways) from our enriched dataset. This yielded a list of real_terms — annotation terms empirically supported in our system.

Step 2: Hierarchical Consolidation of Dictionary Structure
The global_terms structure was then refined using a hybrid strategy:

Minimum viable subcategories: Subcategories (e.g., specific corrosion mechanisms like direct_eet or galvanic_corrosion) were only retained if they contained sufficient real-world support. Sparse subcategories with low or no empirical evidence were eliminated or merged to reduce fragmentation.

Maximum term depth: Within each retained subcategory, we aimed to maximize the number of valid child terms (i.e., biological keywords, gene or enzyme names) to ensure rich annotation coverage.

Semantic reallocation: Unused terms from deprecated categories such as organic_processes and corrosion_keyword_groups were manually reclassified into valid categories where conceptually appropriate (e.g., terms like quorum sensing were reassigned to functional_categories).

This balance of data-driven filtering and semantic grouping led to a revised version of global_terms, maintaining a biologically coherent structure while aligning with the annotation reality of our dataset.

Step 3: Scoring Category Reduction
For downstream scoring and modeling, we focused only on three high-confidence, high-coverage categories:

metal_terms

corrosion_synergies

functional_categories

Other categories (e.g., corrosion_mechanisms, pathway_categories) were retained for network analysis and visualization, but excluded from scoring due to redundancy or sparsity.

# smart_consolidate_terms

In [ ]:
def smart_consolidate_terms(global_terms_list, real_terms):
    """
    Match real_terms to global_terms structure.
    - Retains global_terms structure.
    - Adds unmatched but valid terms to the correct top-level category, in a 'miscellaneous' subcategory.
    - Keeps functional_category scores and justification intact.
    - Collects truly unrecognized terms in manual_review.
    
    Args:
        global_terms_list: List of tuples like [('metal_terms', metal_dict), ...]
        real_terms: Dict {subcat: [terms]} from validation script

    Returns:
        consolidated: Updated global_terms-like dict with only valid real_terms
    """
    from collections import defaultdict
    import copy

    # Start from a deep copy of the base terms
    consolidated = {
        'metal_terms': defaultdict(list),
        'corrosion_synergies': defaultdict(list),
        'functional_categories': defaultdict(lambda: {'terms': [], 'score': 1.0}),
        'corrosion_mechanisms': defaultdict(list),
        'pathway_categories': defaultdict(list),
        'manual_review': defaultdict(list)
    }

    # Preserve all subcategories from global_terms
    term_index = {}  # term_lower → (cat, subcat, score)

    for category_name, cat_dict in global_terms_list:
        for subcat, value in cat_dict.items():
            if isinstance(value, dict) and 'terms' in value:
                score = value.get('score', 1.0)
                for term in value['terms']:
                    term_index[term.lower()] = (category_name, subcat, score)
                    consolidated[category_name][subcat] = {
                        'terms': copy.deepcopy(value['terms']),
                        'score': score,
                        'justification': value.get('justification', '')
                    }
            elif isinstance(value, list):
                for term in value:
                    term_index[term.lower()] = (category_name, subcat, None)
                consolidated[category_name][subcat] = copy.deepcopy(value)

    # Reallocate real terms into this structure
    for subcat, terms in real_terms.items():
        for term in terms:
            tkey = term.lower()
            if tkey in term_index:
                cat, existing_subcat, score = term_index[tkey]

                if cat == 'functional_categories':
                    if term not in consolidated[cat][existing_subcat]['terms']:
                        consolidated[cat][existing_subcat]['terms'].append(term)
                else:
                    if term not in consolidated[cat][existing_subcat]:
                        consolidated[cat][existing_subcat].append(term)

            else:
                # Try to infer best category
                likely_cat = (
                    'functional_categories' if 'ase' in tkey or 'eet' in tkey else
                    'metal_terms' if any(m in tkey for m in ['fe', 'cu', 'zn', 'mn']) else
                    'corrosion_synergies' if '-' in tkey else
                    'pathway_categories', 'corrosion_mechanisms'
                )

                if likely_cat == 'functional_categories':
                    consolidated[likely_cat]['miscellaneous']['terms'].append(term)
                elif likely_cat == 'metal_terms':
                    consolidated[likely_cat]['miscellaneous'].append(term)
                elif likely_cat == 'corrosion_synergies':
                    consolidated[likely_cat]['miscellaneous'].append(term)
                elif likely_cat == 'pathway_categories':
                    consolidated[likely_cat]['miscellaneous'].append(term)
                elif likely_cat == 'corrosion_mechanisms':
                    consolidated[likely_cat]['corrosion_mechanisms'].append(term)
                else:
                    consolidated['manual_review'][subcat].append(term)

    return consolidated


In [ ]:
consolidated = smart_consolidate_terms([
    ('metal_terms', cs.metal_terms),
    ('corrosion_mechanisms', cs.corrosion_mechanisms),
    ('pathway_categories', cs.pathway_categories),
    ('corrosion_synergies', cs.corrosion_synergies),
    ('functional_categories', cs.functional_categories)
], real_terms)



In [ ]:
consolidated

# 5. check_ipath_coverage
While I have made the global term dictionary from the data retrieved, I still muss conciliate the own data that has been possible now to embebed into the eccontri df. The adquisition of the pathways column is a new development and enrich the data. That pathways is going ot be use as golden truth for the retrievals from the databases. The next script calculate the missing terms:

In [6]:
# read the saved parquet file to check if it was saved correctly
ECcontri_Uniprot_path = output_large / 'ECcontri_Uniprot_pathway.parquet'
ECcontri_Uniprot_pathway = pd.read_parquet(ECcontri_Uniprot_path)
#ECcontri_Uniprot_pathway = ECcontri_Uniprot_pathway.sort_values("idx", ascending=True).reset_index(drop=True)

In [7]:
len(ECcontri_Uniprot_pathway["pathways"].unique().tolist())

546

In [8]:
def check_ipath_coverage(ECcontri_Uniprot, functional_categories, pathway_categories, corrosion_mechanisms):
    ''' pathways coming from Picrust2 results are the golden truth will audit the integrity of the dictionaries on global
    terms mainly FC. All FC, mechanisms and pathways are checked to see if they referenced, none category is to be orphaned '''
    # Step 1 golden truth from picrust2
    all_path_terms = set(ECcontri_Uniprot['pathways'].dropna().unique())
    # Step 2 reference terms are in the global terms
    all_functional_terms = set()
    for v in functional_categories.values():
        all_functional_terms.update(v['terms'])
    all_pathway_terms = set()
    for v in pathway_categories.values():
        all_pathway_terms.update(v)
    all_mechanism_terms = set()
    for v in corrosion_mechanisms.values():
        all_mechanism_terms.update(v)
    all_reference_terms = all_functional_terms | all_pathway_terms | all_mechanism_terms 
    # which pathways column terms are not in the dictionaries global terms -Urgent
    missing_in_globalT = all_path_terms - all_reference_terms
    # which global terms are not in the pathway dictionary - no urgent
    unused_globalT = all_reference_terms - all_path_terms
    # which pathway dictionary in global terms are no in FC - to be null
    pathway_dic_not_FC = all_pathway_terms - all_functional_terms
    # which pathways column are not in  fc global terms-Urgent
    pathways_not_in_FunCat = all_path_terms - all_functional_terms 
    # which mechanims in global terms are not in fc
    mech_not_in_FunCat= all_mechanism_terms - all_functional_terms
    # Output
    return {
        'missing_in_globalT': missing_in_globalT,
        'unused_globalT': unused_globalT,
        'pathway_dic_not_FC': pathway_dic_not_FC,
        'pathways_not_in_FunCat': pathways_not_in_FunCat,
        'mech_not_in_FunCat': mech_not_in_FunCat
    }

result_dict = check_ipath_coverage(ECcontri_Uniprot_pathway, cs.functional_categories , cs.pathway_categories, cs.corrosion_mechanisms)  
      

In [9]:
missing_globalT = result_dict['missing_in_globalT']
unused_globalT = result_dict['unused_globalT']  
pathway_dic_not_FC = result_dict['pathway_dic_not_FC']
pathways_not_in_FunCat = result_dict['pathways_not_in_FunCat']
mech_not_in_FunCat = result_dict['mech_not_in_FunCat']

In [12]:
print(f"Missing in global terms: {len(missing_globalT)}")
print(f"Unused global terms: {len(unused_globalT)}")
print(f"Pathway dictionary not in functional categories: {len(pathway_dic_not_FC)}")
print(f"Pathways not in functional categories: {len(pathways_not_in_FunCat)}")
print(f"Mechanisms not in functional categories: {len(mech_not_in_FunCat)}")

Missing in global terms: 5
Unused global terms: 398
Pathway dictionary not in functional categories: 48
Pathways not in functional categories: 5
Mechanisms not in functional categories: 52


In [13]:
mech_not_in_FunCat

{'(p)ppGpp',
 'AHL',
 'AI-2',
 'Mn cycling',
 'Mn-oxide dissolution',
 'acid',
 'acyl homoserine lactone',
 'aerobic respiration',
 'alginate production',
 'autoinducer',
 'biofilm architecture',
 'biofilm dispersal',
 'biofilm matrix',
 'biosorption',
 'birnessite formation',
 'c-di-GMP',
 'carbon catabolite repression',
 'carbonate precipitation',
 'carbonation',
 'carbonic acid',
 'catechol degradation I (meta-claevage pathway)',
 'cellulose production',
 'chelate formation',
 'co-precipitation',
 'eDNA',
 'extracellular DNA',
 'ferric reducction',
 'ferrous oxidation',
 'formate production',
 'gluconic acid',
 'heme biosynthesis',
 'iron oxidation',
 'iron reducction',
 'iron reduction',
 'lactate metabolism',
 'low pH',
 'manganese reduction',
 'matrix metalloprotease',
 'metal binding',
 'metal ligand',
 'metal mobilization',
 'nitrate reduction',
 'nitrite reduction',
 'oxygen consumtion',
 'oxygen depletion',
 'polysulfide',
 'proton generation',
 'pyrolusite formation',
 'quor

The results show that there are 465 pathways from the eccontri that need to be assigned to a functional_category, and doing it by hand is not feasible.Therefore an script has been suggested which create a mapper that links keywords to the functional category. Then a function assign assigns a category to each of the rows(pathway name) if it contains one of those keywords. A priority list would handly cases in which multiple categories claim the name. At the end a manual check of the correct categorisation is done.

In [ ]:
# 1. Build a Prioritized Keyword Mapper
# This dictionary will link a keyword to its category name.

# Define priority: more specific keywords should come first.
priority_keywords = [
    'iron', 'ferric', 'siderophore', 'sulfur', 'sulfide', 'thiosulfate',
    'biofilm', 'alginate', 'peptidoglycan', 'acid', 'acetate', 'lactate', 
    'methionine', 'lysine', 'nitrogen', 'heme', 'metabolism' # General terms last
]

keyword_to_category_map = {}
for category, details in cs.functional_categories.items():
    for term in details['terms']:
        # only add the keyword if it's not already mapped to a higher-priority category
        if term not in keyword_to_category_map:
             keyword_to_category_map[term] = category

In [ ]:
# 2. Create the Automatic Assignment Function 
def assign_category(pathway_name, mapper, priorities):
    """Assigns a category to a pathway name based on prioritized keywords."""
    # Ensure pathway_name is a string and lowercase for matching
    pathway_name_lower = str(pathway_name).lower()
    
    # Check for high-priority keywords first
    for keyword in priorities:
        if keyword in pathway_name_lower:
            # Return the category associated with that keyword
            return mapper.get(keyword)
            
    # If no priority keyword is found, do a general search (slower)
    for keyword, category in mapper.items():
        if keyword in pathway_name_lower:
            return category
            
    # If no match is found at all
    return "Uncategorized"
pathway_name = ECcontri_Uniprot_pathway["pathways"].tolist()
#  Apply the Assignment Function to the DataFrame
ECcontri_Uniprot_pathway['assigned_category'] = ECcontri_Uniprot_pathway['pathways'].apply(lambda x: assign_category(x, keyword_to_category_map, priority_keywords))
#assigned = assign_category(pathway_name, keyword_to_category_map, priority_keywords)
#ECcontri_Uniprot_pathway = ECcontri_Uniprot_pathway.sort_values("idx", ascending=True).reset_index(drop=True)
# drop duplicates
ECcontri_Uniprot_pathway = ECcontri_Uniprot_pathway.drop_duplicates()

In [ ]:
# Display the rows complete with display in screen
pd.set_option("display.max_colwidth", True)
#pd.reset_option('display.max_colwidth')
pd.set_option("display.max_rows", 100)
#pd.reset_option('display.max_columns') 

In [ ]:
ECcontri_Uniprot_pathway[['ec', 'assigned_category', 'pathways']].head(100)

In [ ]:
# 3. Apply the function to uncategorized pathways 
# get the list of pathways from previous function that are NOT in any functional category
pathways_not_in_FunCat = set(pathways_not_in_FunCat)

# Run the automatic assignment
assignments = {pathway: assign_category(pathway, keyword_to_category_map, priority_keywords) 
               for pathway in pathways_not_in_FunCat}


In [ ]:
# Results review the assignments.
assignment_df = pd.DataFrame(list(assignments.items()), columns=['pathways', 'Assigned Category'])

print("\nSample of automatic assignments:")
print(assignment_df.head(10))

print("\nSummary of assignments:")
print(assignment_df['Assigned Category'].value_counts())